# CHE-226 Programming and Data Science
## Assignment 1: Python for Chemical Engineering Calculations

School of Chemical & Materials Engineering, NUST · Fall 2026 · Instructor: Dr. Haider Ejaz

| | |
|---|---|
| **Topics** | T1–T5 (Python basics to lists & tuples) |
| **Assesses** | CLO-1 at Bloom level C2 (understand): code is the tool, explanation earns the marks |
| **Total marks** | 40 (4 problems × 10) |
| **Due** | **Friday 30 October 2026, 11:59 pm** |
| **Submit** | This notebook as `RegNo_A1.ipynb` **and** its PDF export `RegNo_A1.pdf` |

**Name:**

**Registration #:**

### Instructions

- **Tools.** Use only what we covered in T1–T5: variables, arithmetic, f-strings, the `math` module, `if`/`elif`/`else`,
  `for`/`while`, functions (default and keyword arguments, `lambda`, `map`, `filter`), lists, tuples, slicing, `sorted`,
  comprehensions and `zip`. **Do not use NumPy, pandas or Matplotlib** in this assignment.
- **Layout.** Answer each task in the empty code cell under it. Do not edit the *Given* cells. Before you submit,
  choose *Kernel → Restart & Run All* so every output is visible and the notebook runs top to bottom.
- **Interpretation.** Each problem ends with an *Interpretation* cell. Answer every sub-part in full sentences,
  quoting your own numbers with units. Code without interpretation earns at most half the marks for that problem.
- **Style.** Meaningful variable names, a one-line docstring for every function, units in comments, and printed
  tables aligned with f-string widths (e.g. `{P:>8.2f}`).
- **Integrity.** Work individually; matching code between students is treated as copying.

| No. | Chemical engineering context | Ontology topics | Python skills (lecture) | Marks |
|---|---|---|---|---|
| 1 | Real-gas pressure in a CO₂ storage cylinder | T-02.01.01 Gas Laws and Real Gases; T-04.02.02 Equations of State | functions with defaults (T4), `for`/`while` (T3), tuples (T5) | 10 |
| 2 | Mixing BTX streams and checking plant data | T-03.02.01 Single-Unit Balances; T-03.01.02 Composition Measures | `filter`/`lambda` (T4), `zip`, comprehensions, `sorted` with `key` (T5) | 10 |
| 3 | Reaction rate vs temperature in a batch reactor | T-08.01.02 Arrhenius Behaviour; T-08.02.01 Batch Reactor; T-01.04.03 Numerical ODE Solution | `math` (T2), `while` (T3), slicing `ks[1:]` (T5) | 10 |
| 4 | Detecting fouling from heat-exchanger logs | T-06.04.01 Overall Heat Transfer Coefficient; T-06.04.02 LMTD Method; T-01.05.02 Error and Uncertainty Analysis | sentinel & `continue` (T3), tuple unpacking, `min` with `key` (T5) | 10 |

In [ ]:
import math

---
## Problem 1: How far can you trust the ideal-gas law? *(10 marks)*
*Ontology: T-02.01.01 Gas Laws and Real Gases · T-04.02.02 Equations of State · T-03.01.01 Units and Dimensions*

A 50.0 L steel cylinder holds 5.00 kg of CO₂. The cylinder sits outdoors, so its temperature can range from 280 K
on a winter night to 360 K in direct summer sun. Its relief valve opens at 55 bar. The safety engineer wants to know
how much the pressure estimate depends on the gas model used.

$$P_{\text{ideal}} = \frac{nRT}{V} \qquad\qquad P_{\text{vdW}} = \frac{nRT}{V - nb} - \frac{a\,n^2}{V^2} \qquad\qquad Z = \frac{P_{\text{vdW}}}{P_{\text{ideal}}}$$

**Given** (run this cell; do not edit it)

In [ ]:
m = 5000.0        # g of CO2
M = 44.01         # g/mol
V = 50.0          # L
R = 0.08314       # L.bar/(mol.K)
a = 3.640         # L^2.bar/mol^2   (van der Waals, CO2)
b = 0.04267       # L/mol

**(a)** Write two functions, `p_ideal(n, V, T, R=0.08314)` and `p_vdw(n, V, T, a, b, R=0.08314)`, that return pressure in bar. **Estimate** the number of moles `n` and print it to 2 decimal places. **[2]**

In [ ]:
# Your code here

**(b)** For `T` = 280, 300, …, 360 K (use `range`), build a list of tuples `(T, P_ideal, P_vdw, Z)`. Then loop over the list and **summarize** it as an aligned table with a header row (pressures to 2 decimals, `Z` to 3 decimals). **[3]**

In [ ]:
# Your code here

**(c)** Using a `while` loop that starts at 280 K and steps by 1 K, **infer** the first temperature at which the pressure reaches the 55 bar relief setting. Do this once with each model and print both temperatures and the difference between them. **[2]**

In [ ]:
# Your code here

**(d)** **Interpret** your results (answer in the cell below). **[3]**

### Interpretation

(i) How does `Z` change with temperature, and what does $Z<1$ say about the molecules?  
(ii) Which model is the *conservative* choice for sizing the relief valve, and why?  
(iii) Would you trust the ideal-gas law for this cylinder? Support your answer with numbers.

*Write your answer here in full sentences, quoting your own numbers with units.*

---
## Problem 2: Mixing BTX streams: check the data before you balance *(10 marks)*
*Ontology: T-03.02.01 Single-Unit Balances · T-03.01.02 Composition Measures · T-01.09.04 Data Interpretation*

Four streams of benzene (B), toluene (T) and xylene (X) are combined in a mixing tank that runs at steady state
with no reaction. Each record below holds the stream name, its molar flow in kmol/h, and its mole fractions
$[x_B, x_T, x_X]$. One of the records has been typed in wrongly at the plant.

**Given** (run this cell; do not edit it)

In [ ]:
streams = [("S1", 120.0, [0.50, 0.30, 0.20]),
           ("S2",  80.0, [0.20, 0.50, 0.30]),
           ("S3",  50.0, [0.10, 0.20, 0.70]),
           ("S4",  40.0, [0.40, 0.40, 0.28])]
names = ["benzene", "toluene", "xylene"]
M = [78.11, 92.14, 106.17]          # g/mol, same order as names

**(a)** Write `is_valid(x, tol=0.01)`, which returns `True` when the mole fractions in `x` sum to 1 within `tol`. Use `filter` with a `lambda` to keep only the valid streams, and **identify** the names of any that are rejected. **[2]**

In [ ]:
# Your code here

**(b)** Write `mix(stream_list)`, which returns a tuple `(component_flows, total_flow, y)`, where `y` holds the product mole fractions. Use a list comprehension for the component balance $\dot n_i = \sum_j F_j\,x_{ij}$. Run it on the valid streams and **summarize** the results. **[3]**

In [ ]:
# Your code here

**(c)** **Translate** the product into mass flow (kg/h) and mass fractions using `zip(component_flows, M)`. **Classify** the components by mass fraction, largest first, using `sorted(..., key=...)`. **[2]**

In [ ]:
# Your code here

**(d)** **Interpret** your results (you may add a code cell for part ii). **[3]**

In [ ]:
# Part (ii): run mix() on all four streams

### Interpretation

(i) **Compare** the mole and mass compositions. Why does xylene's share grow on a mass basis?  
(ii) Run `mix` on *all four* streams. How far apart are the sum of the component flows and the total feed (kmol/h)? What would this do to a plant balance?  
(iii) In two sentences, **explain** why the validation in (a) must come before any balance is solved.

*Write your answer here in full sentences, quoting your own numbers with units.*

---
## Problem 3: Hotter is faster, but by how much? *(10 marks)*
*Ontology: T-08.01.02 Arrhenius Behaviour · T-08.02.01 Batch Reactor · T-03.03.02 Conversion · T-01.04.03 Numerical ODE Solution*

A liquid-phase reaction A → B is first order and runs in an isothermal batch reactor with $C_{A0} = 2.0$ mol/L.
Its rate constant follows the Arrhenius law, and the batch is finished at 90 % conversion.

$$k = A\,e^{-E_a/RT} \qquad\qquad \frac{dC_A}{dt} = -k\,C_A \qquad\Rightarrow\qquad t_{90} = \frac{-\ln(1-0.9)}{k}$$

**Given:** $A = 5.0\times10^{10}$ min⁻¹, $E_a = 75\,000$ J/mol, $R = 8.314$ J/(mol K). These are the default
arguments of the function in (a).

**(a)** Write `rate_constant(T, A=5.0e10, Ea=75000.0, R=8.314)`, returning $k$ in min⁻¹. For `T` from 300 to 360 K in steps of 10 K, **estimate** $k$ (scientific notation, e.g. `:.3e`) and $t_{90}$ in minutes. **[2]**

In [ ]:
# Your code here

**(b)** Store the $k$ values in a list `ks`. Using `zip(ks, ks[1:])` in a list comprehension, **compare** each $k$ with the next one: print the factor by which $k$ rises per 10 K step, to 2 decimals. **[2]**

In [ ]:
# Your code here

**(c)** Write `simulate_batch(k, dt, X_target=0.9, CA0=2.0)`, which steps the balance forward in time (Euler method: `CA = CA - k*CA*dt`) inside a `while` loop until the conversion $X = (C_{A0}-C_A)/C_{A0}$ reaches `X_target`. It returns `(t, CA)`. At 340 K, run it with `dt` = 2.0, 0.5 and 0.1 min, and **compare** each time with the exact $t_{90}$ (% error). **[3]**

In [ ]:
# Your code here

**(d)** **Interpret** your results (answer in the cell below). **[3]**

### Interpretation

(i) Does the textbook rule "the rate doubles every 10 K" hold here? How does the factor change with temperature?  
(ii) The plant can run one batch per shift (8 h). What is the lowest temperature in your table that meets this target? What might stop you from simply choosing 360 K?  
(iii) **Explain** why the Euler result changes with `dt`, and whether it finishes the batch too early or too late.

*Write your answer here in full sentences, quoting your own numbers with units.*

---
## Problem 4: Is the heat exchanger fouling? *(10 marks)*
*Ontology: T-06.04.01 Overall Heat Transfer Coefficient · T-06.04.02 LMTD Method · T-03.04.01 Enthalpy and Heat Capacity · T-01.05.02 Error and Uncertainty Analysis*

A counter-current exchanger cools hot oil with cooling water. Operators logged the four terminal temperatures (°C)
every hour for a 12-hour shift. A reading of `-999` means the sensor dropped out. The shift engineer suspects that
deposits are building up on the tubes.

$$Q_h = \dot m_h c_{p,h}(T_{h,in}-T_{h,out}) \qquad Q_c = \dot m_c c_{p,c}(T_{c,out}-T_{c,in}) \qquad
\Delta T_{lm} = \frac{\Delta T_1-\Delta T_2}{\ln(\Delta T_1/\Delta T_2)} \qquad U = \frac{Q_h}{A\,\Delta T_{lm}}$$

with $\Delta T_1 = T_{h,in}-T_{c,out}$ and $\Delta T_2 = T_{h,out}-T_{c,in}$ (counter-current).

**Given** (run this cell; do not edit it)

In [ ]:
mh, cph = 2.0, 2.2        # oil:   kg/s, kJ/(kg.K)
mc, cpc = 1.5, 4.18       # water: kg/s, kJ/(kg.K)
Area = 20.0               # m^2
# (hour, Th_in, Th_out, Tc_in, Tc_out)   all in deg C
log = [(1, 150.2, 53.3, 25.1, 93.1), (2, 149.8, 53.3, 24.8, 92.5),
       (3, 150.5, 54.4, 25.3, 92.7), (4, 150.0, 54.4, 25.0, 92.1),
       (5, 149.6, 55.1, 25.2, -999), (6, 150.3, 55.8, 24.9, 91.2),
       (7, 150.1, 56.5, 25.0, 90.7), (8, 149.9, 57.7, 25.4, 90.1),
       (9, 150.4, 58.4, 25.1, 84.2), (10, 150.0, 58.8, 24.7, 88.7),
       (11, 149.7, 59.8, 25.0, 88.1), (12, 150.2, 61.0, 25.2, 87.8)]

**(a)** Write `duty(m, cp, dT)`, returning kW, and `lmtd(dT1, dT2)`, using `math.log`. **Explain** in one comment line why the LMTD, not the arithmetic mean, is used. **[2]**

In [ ]:
# Your code here

**(b)** Loop over `log` and unpack each tuple. Skip any row that contains `-999`, printing a warning with `continue`. For every other hour, **estimate** $Q_h$, $Q_c$, the energy-balance closure error $|Q_h-Q_c|/Q_h\times100$, and $U$ in kW/(m² K). Store each result as a tuple and print an aligned table. **Distinguish** (flag) any hour whose closure error is above 5 %. **[4]**

In [ ]:
# Your code here

**(c)** Keep only the hours with closure ≤ 5 %. Using slicing, **compare** the mean $U$ of the first three and last three of those hours, and report the % drop. Use `min(..., key=...)` to **identify** the hour with the lowest $U$. Take hour 1's $U$ as the clean value, and find the first hour in which $U$ falls below 80 % of it. **[2]**

In [ ]:
# Your code here

**(d)** **Interpret** your results (answer in the cell below). **[2]**

### Interpretation

(i) Is the exchanger fouling? Quote the trend in $U$.  
(ii) Which hour did you flag, and why should it be left out of the $U$ trend rather than "corrected"?  
(iii) When would you recommend taking the exchanger offline for cleaning?

*Write your answer here in full sentences, quoting your own numbers with units.*

---
### Marking guide

| Criterion | Full marks | Partial | Little or none |
|---|---|---|---|
| Code works (tasks a–c) | Runs top to bottom; correct numbers; asked-for constructs used | Minor errors, or the right numbers reached without the requested construct | Does not run, or wrong method |
| Output | Aligned tables, units, sensible decimal places | Correct but hard to read | Raw lists or no units |
| Interpretation (last task) | Answers every sub-part with your own numbers and the engineering reason | Restates numbers without explaining them, or misses a sub-part | Missing, generic, or contradicts the output |

*End of Assignment 1.*